# [SK 07.5 - AI Foundry Agents using Declarative Spec](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python#declarative-spec)
**Note**: `azure-ai-projects==1.1.0b4` and `azure-ai-agents==1.2.0b6` are automatically installed by semantic kernel 1.35.0.<br/>

The AzureAIAgent supports instantiation from a YAML declarative specification. The declarative approach allows you to define the agent's properties, instructions, model configuration, tools, and other options in a single, auditable document. This makes agent composition portable and easily managed across environments.<br/>
A minimal YAML declarative spec might look like the following:
```
type: foundry_agent
name: sk_aifoundry_agent-PYTHON-from-specs
instructions: You are a clever agent
description: This agent answers questions
model:
  id: gpt-4o
  options:
    temperature: 0.4
tools:
  - id: LightsPlugin.get_lights
    type: function
  - id: LightsPlugin.change_state
    type: function
```

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

agent_name = "sk_aifoundry_agent-PYTHON-from-specs"

instructions  = "You are a clever agent"
description   = "This agent answers questions" #  using Bing to provide grounding context.

project_endpoint = os.environ["AIF_STD_PROJECT_ENDPOINT"] # AIF_BAS_PROJECT_ENDPOINT or AIF_STD_PROJECT_ENDPOINT
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://mmoaiswc-01.openai.azure.com/
Project Endpoint: https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.1.0b4
azure-ai-agents library installed version: 1.2.0b6


# 1. Create AI Foundry `AIProjectClient` using [`AzureAIAgent`](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.azureaiagent?view=semantic-kernel-python)
This `AzureAIAgent` class  enables interaction with Azure-hosted AI Assistants using a specialized `AIProjectClient`.<br/>
The agent leverages an AzureAIAgentModel configuration and can optionally override default parameters such as temperature, maximum tokens, or instructions.<br/>
Initialize an AzureAIAgent service by providing at minimum an AIProjectClient and an AzureAIAgentModel

In [2]:
from semantic_kernel.agents import AzureAIAgent
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())

# 2. Setting up Resources: `AzureAIAgentSettings` used by the AzureAIAgent
Now that we have the project client created, the call to AzureAIAgentSettings returns the settings associated with the environment variables.<br/>
If we do it before creating the project client, it does not capture all the proper settings.

In [3]:
from semantic_kernel.agents import AzureAIAgentSettings

aiagent_settings = AzureAIAgentSettings()
aiagent_settings

AzureAIAgentSettings(env_file_path=None, env_file_encoding='utf-8', model_deployment_name='gpt-4o', endpoint='https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2', agent_id=None, bing_connection_id=None, azure_ai_search_connection_id=None, azure_ai_search_index_name=None, api_version=None, deep_research_model=None)

# Define native plugin and planner

In [4]:
class LightsPlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function
   
    def __init__(self):
        self.lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": False},]
 
    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights
 
    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

# Create an AI Foundry Agent

## Define the [YAML specification string](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-architecture?pivots=programming-language-python#agent-types-in-semantic-kernel)
Possible agent types in Semantic Kernel:
- ChatCompletionAgent
- OpenAIAssistantAgent
- AzureAIAgent (**foundry_agent**)
- OpenAIResponsesAgent
- CopilotStudioAgent

In [5]:
spec = f"""
type: foundry_agent
name: {agent_name}
instructions: {instructions}
description: {description}
model:
  id: {aiagent_settings.model_deployment_name}
  options:
    temperature: 0.4
tools:
  - id: LightsPlugin.get_lights
    type: function
  - id: LightsPlugin.change_state
    type: function
"""

print(spec)


type: foundry_agent
name: sk_aifoundry_agent-PYTHON-from-specs
instructions: You are a clever agent
description: This agent answers questions
model:
  id: gpt-4o
  options:
    temperature: 0.4
tools:
  - id: LightsPlugin.get_lights
    type: function
  - id: LightsPlugin.change_state
    type: function



# Creating an SK Agent based on the YAML specs of an AI Foundry agent
In this case, the AI Foundry Agent is created ***on the fly***, and implicitly used to create the wrapping SK agent 

In [6]:
from semantic_kernel.agents import AgentRegistry

agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=spec,
    client=project_client,
    plugins=[LightsPlugin()],
    settings=aiagent_settings,
)

agent

AzureAIAgent(arguments={'temperature': 0.4}, description='This agent answers questions', id='asst_Zggwy73oHweDmC9pqaqhrWgH', instructions='You are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x00000251BF9C9A90>, plugins={'LightsPlugin': KernelPlugin(name='LightsPlugin', description=None, functions={'change_state': KernelFunctionFromMethod(metadata=KernelFunctionMetadata(name='change_state', plugin_name='LightsPlugin', description='Changes the state of the light', parameters=[KernelParameterMetadata(name='id', description=None, default_value=None, type_='int', is_required=True, type_object=<class 'int'>, schema_data={'type': 'integer'}, include_in_function_choices=True), KernelParameterMetadata(name='is_on', description=None, default_value=None, type_='bool', is_required=True, type_object=<class 'bool'>, schema_data={'type': 'boolean'}, include_in_func

# Interacting with an AzureAIAgent
Interaction with the AzureAIAgent is straightforward. The agent maintains the conversation history automatically using a thread.<br/>
The specifics of the Azure AI Agent thread is abstracted away via the AzureAIAgentThread class, which is an implementation of AgentThread.

In [7]:
from semantic_kernel.agents import AzureAIAgentThread
from semantic_kernel.contents import AuthorRole

USER_INPUTS = [
    "Hello", 
    "Please toggle the porch light", 
    "What's the status of all lights?", 
    "Thank you",
]

thread: AzureAIAgentThread = None

try:
    i=0
    for user_input in USER_INPUTS:
        i+=1
        print(f"Message {i} from {AuthorRole.USER}: '{user_input}'")
        # response = await agent.get_response(messages=user_input, thread=thread, arguments=arguments)
        responses = agent.invoke(messages=user_input, thread=thread)
        async for response in responses:
            print(f"Message {i} from {AuthorRole.ASSISTANT}: '{response}'\n")
        thread = response.thread
finally:
    if thread:
        print(f"\nThread <{thread.id}> was created to manage the conversation")

Message 1 from AuthorRole.USER: 'Hello'
Message 1 from AuthorRole.ASSISTANT: 'Hello! How can I assist you today?'

Message 2 from AuthorRole.USER: 'Please toggle the porch light'
Message 2 from AuthorRole.ASSISTANT: 'The porch light has been turned on. Let me know if there's anything else I can help with!'

Message 3 from AuthorRole.USER: 'What's the status of all lights?'
Message 3 from AuthorRole.ASSISTANT: 'Here is the current status of all the lights:

1. **Table Lamp**: Off
2. **Porch Light**: On
3. **Chandelier**: Off

Let me know if you need any changes made!'

Message 4 from AuthorRole.USER: 'Thank you'
Message 4 from AuthorRole.ASSISTANT: 'You're welcome! If you need any further assistance, feel free to ask. Have a great day!'


Thread <thread_wBvDF9YK80VYzQvxjbhNJu1T> was created to manage the conversation


# Teardown

In [8]:
# delete all files
files_to_delete = await project_client.agents.files.list()
files_to_delete_nr = len(files_to_delete.data)

if files_to_delete_nr>0:
    i=0
    print(f"{files_to_delete_nr} files will now be deleted:")
    for f in files_to_delete.data:
        i += 1
        print(f"- File {i} of {files_to_delete_nr}: {f.filename} (id={f.id}) is being deleted...")
        await project_client.agents.files.delete(f.id)
else:
    print("No files to delete")

No files to delete


## Avoiding ***modifying a collection while iterating over it*** for both threads and agents

The code
```
threads_to_delete = project_client.agents.threads.list()
```
returns an async iterator that **lazily** fetches pages of threads.<br/>
But since we're deleting threads as we iterate, the underlying data source is being mutated during iteration. So when the iterator tries to fetch the next page, it hits a missing resource — hence the **ResourceNotFoundError**.<br/><br/>

This is a classic case of *modifying a collection while iterating over it*, which is risky even in synchronous code — and doubly so in async paged APIs.
### The solution
We need to fully materialize the list of threads before deleting anything. That way, the iterator isn’t affected by the deletions

In [9]:
# delete all threads

threads_to_delete = [t async for t in project_client.agents.threads.list()]
i = 0
for t in threads_to_delete:
    i += 1
    print(f"{i} - Thread <{t.id}> is being deleted...")
    await project_client.agents.threads.delete(thread_id=t.id)

1 - Thread <thread_wBvDF9YK80VYzQvxjbhNJu1T> is being deleted...
2 - Thread <thread_qT1NmI7LaPM7RuJqIqsFA1H5> is being deleted...
3 - Thread <thread_yvsMLLTrHLuGOTaP2zO5B4zg> is being deleted...
4 - Thread <thread_Lm0fo3pu0qoYaRWuaCP05sNm> is being deleted...
5 - Thread <thread_KZ2bUlJQSFHjebTuzMIfvsHN> is being deleted...
6 - Thread <thread_kHag9eHu8vYeLqIKnA00Z1BS> is being deleted...
7 - Thread <thread_SskR4lrft2pv8fjb3gl2VRM8> is being deleted...
8 - Thread <thread_8WJf32jAUeNOuzejtw2dSxyu> is being deleted...
9 - Thread <thread_Xognsgf5YoSLYvgLdl1qYDEL> is being deleted...
10 - Thread <thread_fJCnzodmKGVJOw3jAKlZAHOG> is being deleted...
11 - Thread <thread_Dk2IfbvFF7ufFwfwSAHXIuzi> is being deleted...
12 - Thread <thread_1LRwGwIpgcF4rXjfC9iKTur6> is being deleted...
13 - Thread <thread_OcWRTAwsLC1Z68Qnw56BuXlx> is being deleted...
14 - Thread <thread_X9UFbLrfK4htpnUTYKJgGhHT> is being deleted...
15 - Thread <thread_xiBXqV28gtwRNBODa5yNjT0j> is being deleted...
16 - Thread <thread

In [10]:
# delete all agents

agents_to_delete = [a async for a in project_client.agents.list_agents(limit=100)]
i=0
for a in agents_to_delete:
    i += 1
    print(f"{i} - Agent <{a.id}> is being deleted...")
    await project_client.agents.delete_agent(agent_id=a.id)

1 - Agent <asst_Zggwy73oHweDmC9pqaqhrWgH> is being deleted...
2 - Agent <asst_omFm1sWzUnUt4J2Pwhgo5Y70> is being deleted...
3 - Agent <asst_InAXbSxhdmx11T0gnZB3qz4V> is being deleted...
4 - Agent <asst_qYc1L38kQuN3QOkpwTF6ndDF> is being deleted...
5 - Agent <asst_O6sI21bqaGMPpa5Ffpuu2o20> is being deleted...
6 - Agent <asst_ehDFIUlSwCZGw8K3g5qtkKF2> is being deleted...
7 - Agent <asst_RN0uMvRTl9A3Ymkfjcnqgt9P> is being deleted...
8 - Agent <asst_F9SkGpC4ugdqyOPUC0qMFamw> is being deleted...
9 - Agent <asst_dO3Wy0x1urfjNIhmoMm9jpJJ> is being deleted...
10 - Agent <asst_CNHhuK62Ecfsjn13GsRdqvyq> is being deleted...
11 - Agent <asst_9ahmnB2XdJm48xHROKYuKeoV> is being deleted...
12 - Agent <asst_M7aUTkLOOJ2IsNJKFpIr9VEm> is being deleted...
13 - Agent <asst_5OR510L0FQuiukQjtX4wn1O4> is being deleted...
14 - Agent <asst_UVeyKmLv5394S9YeocMvmEzn> is being deleted...
15 - Agent <asst_3y4C625c6hVDU6eBZeExVbMw> is being deleted...
16 - Agent <asst_6GK5jXfcNaNthT4PH9KNSkIH> is being deleted...
1